In [2]:
# Topic Modeling with NMF + LDA, WordNet similarity, Pairwise Jaccard
# Ready for Google Colab / Jupyter. Python 3.8+ recommended.

# ------------------------------------
# Install required dependencies
# ------------------------------------
# !pip install scikit-learn nltk

import numpy as np
import pandas as pd
import re
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import NMF, LatentDirichletAllocation
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk import word_tokenize, download

# ------------------------------------
# Download NLTK resources (IMPORTANT FIX)
# ------------------------------------
download('punkt')
download('punkt_tab')  # <-- FIXES your error
download('wordnet')
download('omw-1.4')
download('stopwords')

# ------------------------------------
# Preprocessing function
# ------------------------------------
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if len(t) > 2 and t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

# ------------------------------------
# Load dataset
# ------------------------------------
print("Loading 20 Newsgroups dataset...")
data = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))
docs_raw = data.data
print(f"Total documents: {len(docs_raw)}")

# ------------------------------------
# Preprocess dataset
# ------------------------------------
print("Preprocessing text...")
docs = [preprocess_text(doc) for doc in docs_raw]

# ------------------------------------
# TF-IDF & Count Vectorization
# ------------------------------------
tfidf_vectorizer = TfidfVectorizer(max_features=1000, max_df=0.95, min_df=5)
tfidf = tfidf_vectorizer.fit_transform(docs)

count_vectorizer = CountVectorizer(max_features=1000, max_df=0.95, min_df=5)
counts = count_vectorizer.fit_transform(docs)

# ------------------------------------
# Task 1: NMF and LDA Topic Modeling
# ------------------------------------
n_topics = 5

nmf = NMF(n_components=n_topics, random_state=42)
nmf_W = nmf.fit_transform(tfidf)
nmf_H = nmf.components_

lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
lda_W = lda.fit_transform(counts)
lda_H = lda.components_

def top_words(model_weights, feature_names, n=10):
    topics = []
    for topic in model_weights:
        top_indices = topic.argsort()[::-1][:n]
        topics.append([feature_names[i] for i in top_indices])
    return topics

nmf_words = top_words(nmf_H, tfidf_vectorizer.get_feature_names_out())
lda_words = top_words(lda_H, count_vectorizer.get_feature_names_out())

print("\n==== NMF vs LDA Topics ====\n")
for i in range(n_topics):
    print(f"Topic {i+1}:")
    print(f" NMF: {nmf_words[i]}")
    print(f" LDA: {lda_words[i]}")
    print()

# ------------------------------------
# Task 2: WordNet Similarity
# ------------------------------------
print("==== WordNet Similarity ====")
word1, word2 = nmf_words[0][0], nmf_words[0][1]
print(f"Comparing words: '{word1}' and '{word2}'")

syn1 = wordnet.synsets(word1)
syn2 = wordnet.synsets(word2)

if syn1 and syn2:
    similarity = syn1[0].wup_similarity(syn2[0])
    print(f"Wu-Palmer Similarity: {similarity}")
else:
    print("One of the words not found in WordNet.")

# ------------------------------------
# Task 3: Jaccard Similarity
# ------------------------------------
print("\n==== Jaccard Similarity ====")

def jaccard(doc1, doc2):
    s1, s2 = set(doc1.split()), set(doc2.split())
    return len(s1 & s2) / len(s1 | s2)

d1, d2, d3 = docs[0], docs[1], docs[2]
print(f"Doc1 vs Doc2: {jaccard(d1, d2)}")
print(f"Doc2 vs Doc3: {jaccard(d2, d3)}")
print(f"Doc1 vs Doc3: {jaccard(d1, d3)}")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Loading 20 Newsgroups dataset...
Total documents: 18846
Preprocessing text...

==== NMF vs LDA Topics ====

Topic 1:
 NMF: ['one', 'would', 'people', 'think', 'god', 'like', 'say', 'right', 'time', 'thing']
 LDA: ['drive', 'window', 'card', 'do', 'problem', 'system', 'one', 'use', 'would', 'work']

Topic 2:
 NMF: ['window', 'file', 'program', 'do', 'use', 'using', 'problem', 'run', 'version', 'application']
 LDA: ['max', 'government', 'state', 'people', 'key', 'gun', 'law', 'right', 'would', 'one']

Topic 3:
 NMF: ['game', 'team', 'year', 'player', 'season', 'last', 'hockey', 'play', 'win', 'fan']
 LDA: ['file', 'program', 'image', 'use', 'system', 'available', 'data', 'space', 'also', 'information']

Topic 4:
 NMF: ['drive', 'card', 'disk', 'scsi', 'system', 'hard', 'controller', 'monitor', 'problem', 'mac']
 LDA: ['one', 'god', 'would', 'people', 'say', 'think', 'know', 'like', 'believe', 'thing']

Topic 5:
 NMF: ['thanks', 'please', 'anyone', 'know', 'mail', 'would', 'advance', 'ema